# 3.0 — Folds chung cho ML + DL

Tạo `folds_5_seed9.csv` (`id, fold`) bằng `StratifiedKFold(5, shuffle=True, random_state=9)` trên thứ tự gốc của `train.csv`,
giống hệt splitter của ML task 1.3–1.5. Mọi notebook 3.x đọc file này thay vì tự chia.

**Input:** dữ liệu cuộc thi + dataset chứa `shared/features.py`. **Output:** `/kaggle/working/folds_5_seed9.csv`.

In [ ]:
import sys
from pathlib import Path

# shared/features.py: local repo or a Kaggle dataset that contains the repo's shared/ folder
cands = [Path("shared/features.py"), Path("../../shared/features.py"), *Path("/kaggle/input").rglob("features.py")]
feat = next(p for p in cands if p.is_file())
sys.path.insert(0, str(feat.parent.resolve()))
import features as F
print("features.py:", feat)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

tr, te, y, train_id, test_id = F.load()
assert len(tr) == F.N_TRAIN and len(te) == F.N_TEST
folds = F.make_folds(train_id, y)

In [ ]:
# Checks: full coverage, 5 folds, stratified, identical to ML's skf.split(tr, y)
assert folds["id"].is_unique and len(folds) == F.N_TRAIN and (folds["fold"] >= 0).all()
summary = folds.assign(is_fraud=y).groupby("fold")["is_fraud"].agg(rows="size", frauds="sum", rate="mean")
print(summary)
assert summary["rows"].between(139_990, 139_994).all()
assert (summary["rate"] - y.mean()).abs().max() < 1e-4

skf = StratifiedKFold(F.N_FOLDS, shuffle=True, random_state=F.SEED)
for f, (_, va) in enumerate(skf.split(tr, y)):
    assert (folds["fold"].to_numpy()[va] == f).all()
print("Folds identical to ML task 1.3-1.5 splitter.")

In [ ]:
out = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("shared/folds")
out.mkdir(parents=True, exist_ok=True)
folds.to_csv(out / "folds_5_seed9.csv", index=False)
print("Saved", out / "folds_5_seed9.csv")